<a href="https://colab.research.google.com/github/ShubhendraP/AgenticAI2026/blob/weekly-classes/Prompt_Testing_OpenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 Prompt Engineering Techniques using OpenAI (LangChain)

This notebook demonstrates **5 core prompt engineering techniques** using the OpenAI API via LangChain.

Understanding how to frame your prompts is one of the most important skills in working with LLMs.
The *same question*, asked in different ways, can produce very different results.

**Techniques covered:**
1. Zero-Shot Prompting
2. Few-Shot Prompting
3. Chain-of-Thought (CoT) Prompting
4. Role-Based Prompting
5. Instruction-Based Prompting

---

In [6]:
#! pip install  langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 1.9 MB/s eta 0:00:00


In [7]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_BASE_URL'] = 'https://openai.vocareum.com/v1'

## 📄 Sample Data – Product Review

We will use a product review as our test data throughout this notebook.
Each prompting technique will be applied to this same text so you can directly compare the results.

In [8]:
# Review text
review_text = """
Product: SmartHeadphones X1
Rating: 4.5/5
Reviewer: Jane Doe
Date: June 15, 2025
Comment: The SmartHeadphones X1 deliver crisp sound and excellent noise cancellation.
Battery life is impressive at 20 hours, but the app could be more user-friendly.
Price: $149.99
"""

## 1️⃣ Zero-Shot Prompting

**What is it?**
You give the model a task with *no examples* — just a clear instruction.

**When to use it:**
When the task is simple and the model is likely to understand the instruction without guidance.

**Watch for:** The model may return slightly inconsistent JSON formatting since it has no example to follow.

In [10]:
#from langchain_openai import ChatOpenAI

prompt = f'''
Extract key-value pairs from the review. Return as JSON.
Review text:
{review_text}'''


llm = ChatOpenAI(
    model_name='gpt-3.5-turbo',
    temperature=0.5
)

response = llm.invoke(prompt)

print("Zero Shot Response",response.content)


Zero Shot Response {
  "Product": "SmartHeadphones X1",
  "Rating": "4.5/5",
  "Reviewer": "Jane Doe",
  "Date": "June 15, 2025",
  "Comment": "The SmartHeadphones X1 deliver crisp sound and excellent noise cancellation. Battery life is impressive at 20 hours, but the app could be more user-friendly.",
  "Price": "$149.99"
}


In [11]:
prompt = f'''
If you have 10 books in your room . You finished up reading 5 . How many books are left in your room
'''


llm = ChatOpenAI(
    model_name='gpt-3.5-turbo',
    temperature=0.5
)

response = llm.invoke(prompt)

print("Zero Shot Response",response.content)


Zero Shot Response There are 5 books left in your room.


Few Shot Response There would be 5 books left in your room.


## 2️⃣ Few-Shot Prompting

**What is it?**
You provide *one or more examples* (input → output pairs) before your actual question.
This teaches the model the exact format you expect.

**When to use it:**
When you need consistent, structured output — like JSON with specific field names.

**Compare with Zero-Shot:** Notice how the output is more consistently formatted here.

In [12]:
prompt = f"""
Extract key-value pairs from the following product review. Return as JSON. Follow this example:
Example:
Text:
Product: Wireless Mouse
Rating: 4/5
Reviewer: John Smith
Date: May 1, 2024
Price: $29.99
Output: ```json
{{
  "Product": "Wireless Mouse",
  "Rating": "4/5",
  "Reviewer": "John Smith",
  "Date": "May 1, 2024",
  "Price": "$29.99"

}}

Text: {review_text}
"""

response = llm.invoke(prompt)

print("Few Shot Response",response.content)

Few Shot Response ```json
{
  "Product": "SmartHeadphones X1",
  "Rating": "4.5/5",
  "Reviewer": "Jane Doe",
  "Date": "June 15, 2025",
  "Comment": "The SmartHeadphones X1 deliver crisp sound and excellent noise cancellation. Battery life is impressive at 20 hours, but the app could be more user-friendly.",
  "Price": "$149.99"
}
```


In [13]:
prompt = f'''
If you have 10 books in your room . You finished up reading 5 . How many books are left in your room.

Example : 15 Books in the room . Finshed up reading 5 Books . Still 15 remaining.
Example : 20 Books in the room . Finshed up reading 10 Books . Still 20 remaining.
'''


llm = ChatOpenAI(
    model_name='gpt-3.5-turbo',
    temperature=0.5
)

response = llm.invoke(prompt)

print("Few Shot Response",response.content)


Few Shot Response There would be 5 books left in your room.


## 3️⃣ Chain-of-Thought (CoT) Prompting

**What is it?**
You ask the model to *reason step-by-step* before giving its final answer.
This mirrors how a human would think through a problem.

**When to use it:**
For tasks that require reasoning, analysis, or multi-step logic — not just extraction.

**Key idea:** Breaking the task into numbered steps significantly improves accuracy on complex questions.

In [14]:
prompt = f'''
If you have 10 books in your room . You finished up reading 5 . How many books are left in your room.

1. Start with the number of book that you have .
2. Since you just read the books they are still there in the room .
3. The total number of books are still the same that you started with
'''


llm = ChatOpenAI(
    model_name='gpt-3.5-turbo',
    temperature=0.0
)

response = llm.invoke(prompt)

print("Chain of Thought ",response.content)


Chain of Thought  Therefore, there are still 10 books left in your room.


In [ ]:
# CoT prompt
prompt = f"""
Given the following product review, answer the question by reasoning step-by-step:
Text: {review_text}
Question: What are the strengths of the product?
Reasoning:
1. Identify the 'Comment' field in the review.
2. Extract positive attributes mentioned in the comment.
3. List these attributes as the strengths.
Answer:
"""

response = llm.invoke(prompt)
result = response.content
print("Chain-of-Thought Result:")
print(result)

Chain-of-Thought Result:
Strengths of the SmartHeadphones X1:
1. Crisp sound quality
2. Excellent noise cancellation
3. Impressive battery life of 20 hours


## 4️⃣ Role-Based Prompting

**What is it?**
You assign a *persona or role* to the model (e.g., 'You are a product analyst').
This shapes the tone, depth, and perspective of the response.

**When to use it:**
When you want domain-specific language, expert analysis, or a particular communication style.

**Try changing the role** to 'customer service agent' or 'marketing copywriter' and see how the response shifts.

In [19]:
# Role-based prompt
prompt = f"""
You are a product recommendation app who do not recommend negative feedback products. Analyze the sentiment of the following product review, focusing on the 'Comment' field. Return 'Positive', 'Negative', or 'Mixed' with a brief explanation.
Text: {review_text}
"""

response = llm.invoke(prompt)

# Extract and print result
result = response.content
print("Role-Based Result:")
print(result)

Role-Based Result:
Mixed

Explanation: The review contains both positive and negative feedback. The reviewer praises the crisp sound, excellent noise cancellation, and impressive battery life of the SmartHeadphones X1. However, they also mention that the app could be more user-friendly, indicating a downside to the product.


## 5️⃣ Instruction-Based Prompting

**What is it?**
You give very *precise, constrained instructions* — specifying exactly how the answer should be formatted or limited.

**When to use it:**
When you need tight control over the output, like a one-sentence answer or a specific value extracted.

**Notice:** The instruction 'in one sentence using exact values from the text' forces the model to be precise and concise.

In [22]:
# Instruction-based prompt
prompt = f"""
Given the following product review, answer the question in one sentence using exact values from the text.
Text: {review_text}
Question: What is the battery life?
Answer:
Who should buy this product?
Ans:
Who should not buy this product?
Ans:
"""

response = llm.invoke(prompt)


# Extract and print result
result = response.content
print("Instruction-Based Result:")
print(result)

Instruction-Based Result:
Question: What is the battery life?
Answer: The battery life is 20 hours.

Who should buy this product?
Ans: People looking for headphones with crisp sound, excellent noise cancellation, and a 20-hour battery life.

Who should not buy this product?
Ans: Individuals who prioritize user-friendly apps over impressive battery life.
